# Comprehensive EDA Pipeline
## Automated Data Profiling & Report Generation

**Supported Formats:** CSV, Excel, JSON, Parquet, Feather, Pickle, TSV

This notebook automatically generates:
- YData Profiling HTML Report
- Sweetviz HTML Report
- AutoViz Visualizations
- D-Tale Interactive Dashboard (in-notebook)

---

## 1. Import Libraries & Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import sys
import warnings
import logging
from pathlib import Path
from datetime import datetime
import chardet
import traceback

warnings.filterwarnings('ignore')

# CONFIGURATION - MODIFY THIS
DATA_FILE = 'your_dataset.csv'  # ← CHANGE THIS TO YOUR FILE
OUTPUT_DIR = 'reports'
CORRELATION_THRESHOLD = 0.5

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✓ Libraries loaded')

## 2. Helper Functions

In [ ]:
def detect_encoding(file_path):
    try:
        with open(file_path, 'rb') as f:
            result = chardet.detect(f.read(10000))
        return result['encoding'] or 'utf-8'
    except:
        return 'utf-8'

def detect_delimiter(file_path, encoding):
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            sample = f.read(10000)
        delimiters = [',', ';', '\t', '|']
        return max(delimiters, key=sample.count)
    except:
        return ','

def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'File not found: {file_path}')
    
    ext = os.path.splitext(file_path)[1].lower()
    
    try:
        if ext in ['.csv', '.tsv']:
            enc = detect_encoding(file_path)
            delim = detect_delimiter(file_path, enc)
            df = pd.read_csv(file_path, encoding=enc, sep=delim)
        elif ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
        elif ext == '.json':
            df = pd.read_json(file_path)
        elif ext == '.parquet':
            df = pd.read_parquet(file_path)
        elif ext == '.feather':
            df = pd.read_feather(file_path)
        elif ext in ['.pkl', '.pickle']:
            df = pd.read_pickle(file_path)
        else:
            raise ValueError(f'Unsupported format: {ext}')
        return df
    except Exception as e:
        print(f'Error loading data: {e}')
        raise

print('✓ Helper functions ready')

## 3. Load Dataset

In [ ]:
start_time = datetime.now()

try:
    df = load_data(DATA_FILE)
    dataset_name = os.path.splitext(os.path.basename(DATA_FILE))[0]
    dataset_output_dir = os.path.join(OUTPUT_DIR, dataset_name)
    Path(dataset_output_dir).mkdir(parents=True, exist_ok=True)
    
    print(f'✓ Data loaded successfully!')
    print(f'  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
    print(f'  Output: {dataset_output_dir}')
except Exception as e:
    print(f'✗ Failed to load: {e}')
    sys.exit(1)

## 4. Dataset Overview

In [ ]:
print('\n' + '='*80)
print('DATASET OVERVIEW')
print('='*80)
print(f'\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\nColumn Information:')
print(df.dtypes)

print('\nFirst 5 rows:')
df.head()

## 5. Data Types & Missing Values

In [ ]:
print('\n' + '='*80)
print('DATA TYPES & MISSING VALUES')
print('='*80)

info_df = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.values,
    'Non-Null': df.count().values,
    'Null': df.isnull().sum().values,
    'Null %': (df.isnull().sum().values / len(df) * 100).round(2)
})

print('\n', info_df.to_string(index=False))
print(f'\nTotal Missing Values: {df.isnull().sum().sum()}')

## 6. Numerical Statistics

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numerical_cols:
    print('\n' + '='*80)
    print('NUMERICAL STATISTICS')
    print('='*80)
    stats = df[numerical_cols].describe(include='all').T
    stats['skew'] = df[numerical_cols].skew()
    stats['kurtosis'] = df[numerical_cols].kurtosis()
    print('\n', stats.round(4))
else:
    print('\nNo numerical columns found.')

## 7. Categorical Statistics

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_cols:
    print('\n' + '='*80)
    print('CATEGORICAL STATISTICS')
    print('='*80)
    for col in categorical_cols:
        print(f'\n{col}:')
        print(f'  Unique: {df[col].nunique()}')
        print(f'  Top 5 values:')
        print(df[col].value_counts().head().to_string())
else:
    print('\nNo categorical columns found.')

## 8. Correlation Matrix

In [ ]:
if len(numerical_cols) > 1:
    print('\n' + '='*80)
    print('CORRELATION MATRIX')
    print('='*80)
    corr_matrix = df[numerical_cols].corr()
    print('\nFull Correlation Matrix:')
    print(corr_matrix.round(3))
    
    print(f'\nHigh Correlations (abs > {CORRELATION_THRESHOLD}):')
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > CORRELATION_THRESHOLD:
                high_corr.append({
                    'Column 1': corr_matrix.columns[i],
                    'Column 2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    if high_corr:
        print(pd.DataFrame(high_corr).to_string(index=False))
    else:
        print(f'No correlations above {CORRELATION_THRESHOLD} threshold.')
else:
    print('\nNot enough numerical columns for correlation analysis.')

## 9. Generate YData Profiling Report

In [ ]:
print('\n' + '='*80)
print('YDATA PROFILING REPORT')
print('='*80)

try:
    from ydata_profiling import ProfileReport
    print('\nGenerating profile (this may take 1-2 minutes)...')
    
    profile = ProfileReport(
        df,
        title=f'YData Profile - {dataset_name}',
        explorative=True,
        progress_bar=True
    )
    
    ydata_file = os.path.join(dataset_output_dir, 'ydata_profiling_report.html')
    profile.to_file(ydata_file)
    
    print(f'✓ YData report saved: {ydata_file}')
    print(f'  File size: {os.path.getsize(ydata_file) / 1024**2:.2f} MB')

except ImportError:
    print('⚠ YData Profiling not installed.')
    print('  Install: pip install ydata-profiling')
except Exception as e:
    print(f'⚠ Error generating YData report: {e}')

## 10. Generate Sweetviz Report

In [ ]:
print('\n' + '='*80)
print('SWEETVIZ REPORT')
print('='*80)

try:
    import sweetviz as sv
    print('\nGenerating Sweetviz report...')
    
    my_report = sv.analyze(df, target_name=None)
    
    sweetviz_file = os.path.join(dataset_output_dir, 'sweetviz_report.html')
    my_report.show(sweetviz_file, open_browser=False)
    
    print(f'✓ Sweetviz report saved: {sweetviz_file}')
    print(f'  File size: {os.path.getsize(sweetviz_file) / 1024**2:.2f} MB')

except ImportError:
    print('⚠ Sweetviz not installed.')
    print('  Install: pip install sweetviz')
except Exception as e:
    print(f'⚠ Error generating Sweetviz report: {e}')

## 11. Generate AutoViz Visualizations

In [ ]:
print('\n' + '='*80)
print('AUTOVIZ VISUALIZATIONS')
print('='*80)

try:
    from autoviz.AutoViz_Class import AutoViz_Class
    print('\nGenerating AutoViz plots...')
    
    autoviz_dir = os.path.join(dataset_output_dir, 'autoviz_plots')
    Path(autoviz_dir).mkdir(parents=True, exist_ok=True)
    
    AV = AutoViz_Class()
    AV.fit_transform(
        DATA_FILE,
        sep=',',
        depVar='',
        verbose=0,
        chart_format='svg',
        max_rows_analyzed=len(df),
        max_cols_analyzed=len(df.columns)
    )
    
    print(f'✓ AutoViz plots saved to: {autoviz_dir}')
    svg_files = [f for f in os.listdir(autoviz_dir) if f.endswith('.svg')]
    print(f'  Generated {len(svg_files)} plots')

except ImportError:
    print('⚠ AutoViz not installed.')
    print('  Install: pip install autoviz')
except Exception as e:
    print(f'⚠ Error generating AutoViz plots: {e}')

## 12. Launch D-Tale Interactive Dashboard

In [ ]:
print('\n' + '='*80)
print('D-TALE INTERACTIVE DASHBOARD')
print('='*80)

try:
    import dtale
    print('\nLaunching D-Tale dashboard...')
    
    d = dtale.show(df, open_browser=False)
    dtale_url = d.open_browser()
    
    print(f'✓ D-Tale is running!')
    print(f'  URL: {dtale_url}')
    print('\nD-Tale Features:')
    print('  - Interactive data exploration')
    print('  - Column statistics')
    print('  - Filtering & sorting')
    print('  - Visualization tools')
    print('\nNote: Keep this notebook running to access D-Tale.')
    print('\nTo stop D-Tale, uncomment and run the cleanup cell below.')
    
    dtale_instance = d

except ImportError:
    print('⚠ D-Tale not installed.')
    print('  Install: pip install dtale')
except Exception as e:
    print(f'⚠ Error launching D-Tale: {e}')

## 13. Execution Summary

In [ ]:
end_time = datetime.now()
execution_time = (end_time - start_time).total_seconds()

print('\n' + '='*80)
print('EXECUTION COMPLETE')
print('='*80)

print(f'\n✓ EDA Pipeline finished successfully!')
print(f'  Execution Time: {execution_time:.2f} seconds')
print(f'  Output Directory: {dataset_output_dir}')

# List generated files
print(f'\nGenerated Reports:')
for item in os.listdir(dataset_output_dir):
    item_path = os.path.join(dataset_output_dir, item)
    if os.path.isfile(item_path):
        size_mb = os.path.getsize(item_path) / 1024**2
        print(f'  ✓ {item} ({size_mb:.2f} MB)')
    elif os.path.isdir(item_path):
        file_count = len([f for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f))])
        print(f'  ✓ {item}/ ({file_count} files)')

print(f'\nOpen these reports in a web browser to explore the data!')

## 14. Optional Cleanup

In [ ]:
# Uncomment to stop D-Tale
# if 'dtale_instance' in locals():
#     dtale_instance.kill()
#     print('D-Tale dashboard stopped.')